# Sinh biểu đồ coherence / diversity / runtime / robustness — CafeBERT full benchmark

Notebook này đọc trực tiếp `benchmark/cafebert_full/reference/full_results.csv` (480 dòng thật: 4 corpus × 6 mô hình × 5 giá trị k × 4 seed, đã audit `PASS` — xem `reference/FULL_MULTISEED_AUDIT.md`) và vẽ 4 nhóm biểu đồ, cùng phong cách với `NLP/main/runtime.png`/`topic_coherence.png`/`topic_diversity.png` (đường màu theo mô hình, lưới subplot theo corpus).

**Bạn đã có đủ số liệu để làm việc này** — không cần chạy lại benchmark. Cột dữ liệu đã có sẵn:
- **Coherence** → cột `wec_in` (WEC-in, coherence chính theo protocol S³).
- **Diversity** → cột `topic_diversity`.
- **Runtime** → 3 cột `fit_seconds` / `pipeline_seconds` / `total_cold_seconds` (xem giải thích ở mục Runtime bên dưới — KHÔNG dùng lẫn lộn 3 cột này).
- **Robustness** → cột `c_npmi` (Gensim C_NPMI, README ghi rõ: dùng để kiểm tra không đồng thuận, không dùng để chọn "người thắng").

**Khác biệt quan trọng so với `NLP/main/runtime.png`**: bộ đó có lưới 5 dataset × 4 encoder (vì paper gốc so nhiều encoder tiếng Anh). Benchmark CafeBERT này chỉ dùng **một encoder** (CafeBERT) cho S³/BERTopic, nên lưới ở đây là **2×2 theo 4 corpus tiếng Việt** (không có chiều encoder) — đúng với dữ liệu thật đang có, không bịa thêm chiều so sánh không tồn tại.

**Yêu cầu môi trường**: máy này đã có `pandas`/`matplotlib` trong `.venv`, nhưng **chưa có `jupyter`/`ipykernel`** để chạy notebook. Cài trước khi Run All:
```powershell
.venv\Scripts\python.exe -m pip install jupyter ipykernel
```
Hoặc mở file này trong VSCode và chọn kernel `.venv` — VSCode sẽ tự đề nghị cài `ipykernel` nếu thiếu.

Biểu đồ xuất ra `benchmark/cafebert_full/notebook_charts/` (thư mục riêng, không ghi đè lên các PNG đã có sẵn trong `reference/`).

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(marker: str = "benchmark/cafebert_full/reference/full_results.csv") -> Path:
    """Works whether the notebook's cwd is the repo root or benchmark/cafebert_full/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Khong tim thay '{marker}' tu {Path.cwd()} hay bat ky thu muc cha nao. "
        "Hay chay notebook nay tu repo root (E:/Development/NLP_S3) hoac tu benchmark/cafebert_full/."
    )


ROOT = find_repo_root()
REFERENCE_DIR = ROOT / "benchmark" / "cafebert_full" / "reference"
CSV_PATH = REFERENCE_DIR / "full_results.csv"
OUTPUT_DIR = ROOT / "benchmark" / "cafebert_full" / "notebook_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {ROOT}")
print(f"Doc du lieu tu: {CSV_PATH}")
print(f"Xuat bieu do vao: {OUTPUT_DIR}")

In [ ]:
# Cung tu vung mau/nhan voi benchmark/cafebert_full/generate_cafebert_full_report.py
# de bieu do trong notebook nay "cung nha" voi cac PNG da co san trong reference/.

TOPIC_COUNTS = [10, 20, 30, 40, 50]
SEEDS = [11, 29, 42, 47]

MODEL_ORDER = ["s3_axial", "s3_angular", "s3_combined", "lda", "nmf", "bertopic_kmeans"]
MODEL_LABELS = {
    "s3_axial": "S\u00b3 axial",
    "s3_angular": "S\u00b3 angular",
    "s3_combined": "S\u00b3 combined",
    "lda": "LDA",
    "nmf": "NMF",
    "bertopic_kmeans": "BERTopic + UMAP + KMeans",
}
# Xanh duong/tim/xanh la = 3 bien the S3 (nhom chinh); nau/xam/do = 3 baseline.
# Do dam/nhat KHONG chi den tu hex ma tu alpha o ham ve (S3 = duong net, khong
# trong suot; baseline = nhat/mo) -- xem plot_metric_grid ben duoi.
COLORS = {
    "S\u00b3 axial": "#1d4ed8",
    "S\u00b3 angular": "#7c3aed",
    "S\u00b3 combined": "#15803d",
    "LDA": "#b45309",
    "NMF": "#64748b",
    "BERTopic + UMAP + KMeans": "#dc2626",
}

CORPUS_ORDER = ["vietnamese-news", "visfd", "vi-medical", "vntc-it"]
CORPUS_LABELS = {
    "vietnamese-news": "Vietnamese-news",
    "visfd": "UIT-ViSFD",
    "vi-medical": "ViMedical Disease",
    "vntc-it": "VNTC-CNTT",
}

In [ ]:
frame = pd.read_csv(CSV_PATH)
frame["method"] = pd.Categorical(
    frame["model"].map(MODEL_LABELS), [MODEL_LABELS[m] for m in MODEL_ORDER], ordered=True
)
frame["corpus_label"] = pd.Categorical(
    frame["corpus"].map(CORPUS_LABELS), [CORPUS_LABELS[c] for c in CORPUS_ORDER], ordered=True
)

expected_rows = len(CORPUS_ORDER) * len(MODEL_ORDER) * len(SEEDS) * len(TOPIC_COUNTS)
assert len(frame) == expected_rows, f"Ky vong {expected_rows} dong, doc duoc {len(frame)}"
assert frame["status"].eq("ok").all(), "Co dong status != 'ok', kiem tra lai truoc khi ve bieu do"
print(f"OK: {len(frame)} dong, tat ca status=ok.")
frame.head(3)

## Hàm vẽ dùng chung

Kỹ thuật để có màu sắc dễ nhìn giống `NLP/main/runtime.png`, và để **3 đường S³ nổi bật đậm, 3 đường baseline chìm nhạt** (đúng yêu cầu):
1. **`plt.style.use("seaborn-v0_8-whitegrid")`** — nền lưới xám nhạt, bỏ khung trên/phải (`ax.spines`) → biểu đồ trông "gọn", không rối.
2. **Độ đậm/nhạt = `alpha`, không chỉ là chọn màu**: S³ vẽ với `alpha=1.0` (nét đặc, không trong suốt); 3 baseline vẽ với `alpha` thấp hơn hẳn (`~0.45`) — trên nền trắng, alpha thấp tự động làm màu trông nhạt/mờ đi mà không cần đổi hex. Đây là cách chắc chắn nhất để "đậm/nhạt" đúng ý, thay vì chỉ chọn tông màu sáng/tối (dễ vẫn nổi ngang nhau nếu độ bão hoà giống nhau).
3. **Đường S³ vẽ dày hơn rõ rệt** (`linewidth` lớn hơn, marker to hơn, `zorder` cao hơn để luôn nằm trên baseline khi hai đường chồng nhau).
4. **Dải mean ± SD**: S³ có dải mờ đậm hơn một chút (`alpha=0.12`) so với baseline (`alpha=0.05`) — tránh dải mờ của baseline che mất đường S³ đậm.
5. **Lưới subplot theo corpus** (2×2), 1 legend chung ở trên cùng thay vì lặp lại legend ở từng ô.
6. **Thang log cho runtime** (`ax.set_yscale("log")`) — bắt buộc khi các phương pháp chênh nhau hàng chục/hàng trăm lần, nếu không đường nhanh sẽ dí sát trục hoành và không đọc được.

In [ ]:
def plot_metric_grid(
    frame: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    output_path: Path,
    log_scale: bool = False,
    s3_linewidth: float = 2.6,
    baseline_linewidth: float = 1.3,
    s3_line_alpha: float = 1.0,
    baseline_line_alpha: float = 0.45,
    s3_fill_alpha: float = 0.12,
    baseline_fill_alpha: float = 0.05,
) -> Path:
    """2x2 grid (1 subplot / corpus), 1 duong mau / mo hinh, dai mean+-SD.
    S3 = net dam (alpha=1.0, day, marker to); baseline = net nhat/mo (alpha thap)
    de S3 luon noi bat truoc, dung cach 'dam/nhat' theo yeu cau. Luu PNG va
    tra ve duong dan de kiem tra nhanh."""
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.rcParams.update({"font.size": 10})
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.6), dpi=170, sharex=True)

    for ax, corpus in zip(axes.flat, CORPUS_ORDER, strict=True):
        part = frame.loc[frame["corpus"] == corpus]
        aggregate = (
            part.groupby(["method", "n_topics"], observed=True)[metric]
            .agg(["mean", "std"])
            .reset_index()
        )
        # Ve baseline truoc (nhat, o duoi), S3 ve sau (dam, de len tren) --
        # zorder da xu ly thu tu ve nhung ve baseline truoc van giup tranh
        # canh S3 bi net baseline de len ria.
        draw_order = [m for m in MODEL_ORDER if not m.startswith("s3_")] + [
            m for m in MODEL_ORDER if m.startswith("s3_")
        ]
        for model in draw_order:
            method = MODEL_LABELS[model]
            values = aggregate.loc[aggregate["method"] == method].sort_values("n_topics")
            is_s3 = model.startswith("s3_")
            line_alpha = s3_line_alpha if is_s3 else baseline_line_alpha
            fill_alpha = s3_fill_alpha if is_s3 else baseline_fill_alpha
            ax.plot(
                values["n_topics"],
                values["mean"],
                color=COLORS[method],
                alpha=line_alpha,
                marker="o",
                markersize=4.2 if is_s3 else 3.0,
                linewidth=s3_linewidth if is_s3 else baseline_linewidth,
                label=method,
                zorder=3 if is_s3 else 2,
            )
            sd = values["std"].fillna(0)
            ax.fill_between(
                values["n_topics"],
                values["mean"] - sd,
                values["mean"] + sd,
                color=COLORS[method],
                alpha=fill_alpha,
                zorder=1,
            )
        if log_scale:
            ax.set_yscale("log")
        ax.set_title(CORPUS_LABELS[corpus], loc="left", fontweight="bold")
        ax.set_xticks(TOPIC_COUNTS)
        ax.set_xlabel("So topic (k)")
        ax.set_ylabel(ylabel)
        ax.spines[["top", "right"]].set_visible(False)

    # Legend theo dung MODEL_ORDER (khong theo thu tu ve) de S3 luon hien truoc.
    handles_by_label = dict(zip(*axes.flat[0].get_legend_handles_labels()[::-1]))
    ordered_labels = [MODEL_LABELS[m] for m in MODEL_ORDER]
    handles = [handles_by_label[label] for label in ordered_labels]
    fig.legend(handles, ordered_labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.04), frameon=False)
    fig.suptitle(title, y=1.10, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(output_path, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Da luu: {output_path}")
    return output_path

## 1. Coherence (WEC-in)

Metric coherence chính theo protocol S³ (cosine trung bình giữa các cặp top-term trong từng topic, Word2Vec huấn luyện trên chính corpus). Giá trị càng cao càng tốt.

In [ ]:
plot_metric_grid(
    frame,
    metric="wec_in",
    ylabel="WEC-in",
    title="Coherence (WEC-in) theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "topic_coherence.png",
)

## 2. Diversity

Tỉ lệ top-term không trùng lặp trên toàn bộ topic. Giá trị cao **không** đồng nghĩa coherence cao (một mô hình có thể đa dạng nhưng lộn xộn) — nên luôn đọc cùng biểu đồ coherence ở trên, không đọc riêng lẻ.

In [ ]:
plot_metric_grid(
    frame,
    metric="topic_diversity",
    ylabel="Topic diversity",
    title="Diversity theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "topic_diversity.png",
)

## 3. Runtime — 2 phiên bản, KHÔNG gộp lẫn

`README.md` của benchmark này nhấn mạnh: *"A fit-only result must not be presented as end-to-end runtime."* Vì vậy notebook vẽ **2 biểu đồ tách biệt**, đều ở thang log (chênh lệch giữa các phương pháp lên tới hàng chục/hàng trăm lần):

- **`fit_seconds`** (fit-only, warm) — chỉ thời gian fit mô hình *sau khi* embedding/CountVectorizer đã sẵn sàng. Đây là con số công bằng nhất để so sánh **tốc độ thuật toán** giữa S³/LDA/NMF/BERTopic, không lẫn chi phí encode CafeBERT.
- **`pipeline_seconds`** (representation cold-reference + fit) — cộng thêm thời gian encode CafeBERT (cho S³/BERTopic) hoặc CountVectorizer (cho LDA/NMF). Đây là con số gần với trải nghiệm thực tế "từ văn bản thô tới ra topic", nhưng **không phải ablation encoder** vì LDA/NMF vốn dĩ không dùng CafeBERT (khác lớp mô hình, không phải cùng encoder chạy nhanh hơn).

In [ ]:
plot_metric_grid(
    frame,
    metric="fit_seconds",
    ylabel="Fit-only, warm (giay, thang log)",
    title="Runtime fit-only theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "runtime_fit_only.png",
    log_scale=True,
)

In [ ]:
plot_metric_grid(
    frame,
    metric="pipeline_seconds",
    ylabel="Pipeline cold-reference (giay, thang log)",
    title="Runtime pipeline (bieu dien + fit) theo so topic, tren 4 corpus tieng Viet",
    output_path=OUTPUT_DIR / "runtime_pipeline.png",
    log_scale=True,
)

## 4. Robustness (C_NPMI)

Gensim C_NPMI — metric coherence phụ dựa trên đồng xuất hiện từ, dùng làm **phép kiểm tra không đồng thuận** với WEC-in (thứ hạng có thể khác nhau, không sao). Theo `README.md`: *"C_NPMI ... is not used to select a winner."* Vẽ ở đây để xem WEC-in có "đứng vững" khi đổi cách đo coherence hay không, không dùng để tuyên bố mô hình nào thắng.

In [ ]:
plot_metric_grid(
    frame,
    metric="c_npmi",
    ylabel="C_NPMI (robustness, khong dung chon winner)",
    title="Robustness (C_NPMI) theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "robustness_c_npmi.png",
)

## Phụ lục: số ô WEC-in mà một biến thể S³ thắng

Bảng tóm tắt nhanh (không phải biểu đồ) — hữu ích nếu bạn muốn trích một câu số liệu vào báo cáo, tương tự cách `S3_CAFEBERT_FULL_VIETNAMESE_REPORT.md` đã viết.

In [ ]:
def count_wec_wins(frame: pd.DataFrame, seed_scope: list[int]) -> tuple[int, int, dict[str, int]]:
    scoped = frame.loc[frame["seed"].isin(seed_scope)]
    total = 0
    s3_wins = 0
    corpus_wins = {corpus: 0 for corpus in CORPUS_ORDER}
    for (corpus, _seed, _k), group in scoped.groupby(["corpus", "seed", "n_topics"], observed=True):
        max_score = group["wec_in"].max()
        winners = set(group.loc[group["wec_in"].eq(max_score), "model"])
        total += 1
        if any(model.startswith("s3_") for model in winners):
            s3_wins += 1
            corpus_wins[corpus] += 1
    return s3_wins, total, corpus_wins


s3_wins, total_cells, corpus_wins = count_wec_wins(frame, SEEDS)
print(f"S3 dan dau WEC-in trong {s3_wins}/{total_cells} o corpus x seed x k.")
for corpus in CORPUS_ORDER:
    print(f"  {CORPUS_LABELS[corpus]:20s}: {corpus_wins[corpus]}/20")

## Dùng biểu đồ trong `report/paper.tex`

Các PNG xuất ra `benchmark/cafebert_full/notebook_charts/`. Muốn đưa vào báo cáo ACL, copy sang `report/figures/` rồi `\includegraphics{figures/<ten_file>.png}`, ví dụ:
```
cp benchmark/cafebert_full/notebook_charts/topic_coherence.png report/figures/vn_coherence.png
```
(đổi tên để không đè lên `report/figures/topic_coherence.png` hiện có — file đó là ảnh tái hiện từ **paper gốc tiếng Anh**, còn ảnh mới này là **của nhóm, trên 4 corpus tiếng Việt** — hai thứ khác nhau, đừng nhầm khi viết caption).